In [ ]:
# Code modified from some defined functions  from
# https://github.com/czbiohub-sf/comparison-RNAVelo/blob/main/method-agreement/compute_median_vector_cosinesim_df.py

In [2]:
import pandas as pd
import scanpy as sc
import anndata
import scvelo as scv
import numpy as np
import os

In [ ]:

def compute_cosine_similarities(matrix, fixed_matrix):
    dot_products = np.sum(matrix * fixed_matrix, axis=1) 
    norms = np.linalg.norm(matrix, axis=1) * np.linalg.norm(fixed_matrix, axis=1) 
    return dot_products / norms
def scale_transition_matrices(adata, key1):
    A = adata.uns[key1].copy()
    A_norm = scv.utils.get_transition_matrix(adata=adata, vgraph=A)
    return A_norm
def align_matrices(matrices, cell_ids):
    # Initialize a DataFrame from the first matrix to start the alignment
    reference_df = pd.DataFrame(matrices[0], index=cell_ids[0], columns=cell_ids[0]) 
    aligned_matrices = [reference_df.values]
    
    # Align each subsequent matrix to the reference cell ID order
    for matrix, ids in zip(matrices[1:], cell_ids[1:]):
        df = pd.DataFrame(matrix, index=ids, columns=ids)
        aligned_df = df.reindex(index=reference_df.index, columns=reference_df.index) 
        aligned_matrices.append(aligned_df.values)
    
    return np.array(aligned_matrices), reference_df.index.tolist()
def compute_median_matrix(matrices):
    # Stack the aligned matrices and compute the median across the first axis
    stacked_matrices = np.stack(matrices, axis=0) 
    median_matrix = np.median(stacked_matrices, axis=0) 
    return median_matrix

In [ ]:
data_dir = '/data_path/velocity_cosistency_1st_batch/'
save_dir="/save_path/methods_agree_1st_batch/A2/csv/" 

In [ ]:

datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina'] 
methods=['velocyto','scVelo_stochastic','scVelo_dynamic','MultiVelo','veloAE','veloVI','VeloVAE','uniTvelo','Deepvelo2024','pyro-velocity','cell2fate','latentvelo'] #'Dynamo','cellDancer',
ind=np.arange(0,6) #
method_ind=np.arange(0,12) 
for i in ind:
    #i=0
    print(datasets[i])
    matrices = []
    cell_ids = []
    os.chdir(f'/data_path/velocity_cosistency_1st_batch/{datasets[i]}')
    g_method_dir="/data_path/velocity_cosistency_1st_batch/g_3_method/"
    for j in method_ind:
        if j==10 or j==11 or j==12:
            adata=sc.read_h5ad(g_method_dir+datasets[i]+"/"+f'{datasets[i]}_{methods[j]}_consistency_score.h5ad')
        else:
            adata=sc.read_h5ad(f'{datasets[i]}_{methods[j]}_consistency_score.h5ad')
        ##vkey
        if methods[j]=="VeloVAE":
            adata.uns['velocity_graph']=adata.uns['vae_velocity_graph']
        elif methods[j]=="veloAE":
            adata.uns['velocity_graph']=adata.uns['new_velocity_graph']
        elif methods[j]=="MultiVelo":
            adata.uns['velocity_graph']=adata.uns['velo_s_norm_graph']
        elif methods[j]=="pyro_velocity":
            adata.uns['velocity_graph']=adata.uns['velocity_pyro_graph']
        elif methods[j]=="cell2fate":
            adata.uns['velocity_graph']=adata.uns['Velocity_graph']
        elif methods[j]=="latentvelo":
            adata.uns['velocity_graph']=adata.uns['spliced_velocity_graph']
        else:
            adata.uns['velocity_graph']=adata.uns['velocity_graph']
        ##
        matrix = scale_transition_matrices(adata, 'velocity_graph').toarray() 
        ids = adata.obs.index.tolist().copy() 
        matrices.append(matrix) 
        cell_ids.append(ids)
    aligned_matrices, common_cell_ids = align_matrices(matrices, cell_ids) 
    median_matrix = compute_median_matrix(aligned_matrices) 
    print("Median Matrix Shape:", median_matrix.shape) 
    print("Common Cell IDs:", common_cell_ids[:10])  # Show the first 10
    cosine_similarities = {method: compute_cosine_similarities(matrix, median_matrix) for method, matrix in zip(methods, aligned_matrices)}
    # Construct the DataFrame
    cosine_similarity_df = pd.DataFrame(cosine_similarities, index=common_cell_ids) 
    print(cosine_similarity_df.head())
    cosine_similarity_df.to_csv(save_dir+f'{datasets[i]}_cosine_similar_medianvector.csv')

In [5]:
cosine_similarity_df.to_csv(save_dir+f'{datasets[i]}_cosine_similar_medianvector.csv')